In [8]:
import geogridfusion
import pvdeg
import pvlib

In [ ]:
# downloads container from docker
geogridfusion.run_container(accept_docker=None)

=== GeoGridFusion will use Docker ===
- Action: PULL & RUN a container image
- Image:  postgis/postgis:14-3.5
- Source: Docker Hub
- Port:   127.0.0.1:5433 -> container:5432
- Data:   named volume "pgdata"

This will download data from the internet and start a background service.
geogridfusion-pg already running (id=7d1f11a66f06)


<Container: 7d1f11a66f06>

In [11]:
fusion = geogridfusion.geogridfusionStore()
fusion.connect()

PostgreSQL connection established after 0.02 seconds.


In [4]:
sw, sm = pvlib.iotools.get_solrad(station="abq", start="2022-01-01", end="2022-01-05")

In [14]:
fusion.store_single(
    weather_df=pvdeg.weather.map_weather(sw),
    meta=pvdeg.weather.map_meta(sm),
    source_name="solrad",
    tmy=False,
)

duplicate file detected, skipping insert
metadata of duplicate file {'station': 'abq', 'filenames': ['abq/2022/abq22001.dat', 'abq/2022/abq22002.dat', 'abq/2022/abq22003.dat', 'abq/2022/abq22004.dat', 'abq/2022/abq22005.dat'], 'station_name': 'Albuquerque', 'latitude': 35.03796, 'longitude': -106.62211, 'altitude': 1617.0, 'TZ': -7}


In [12]:
weather, meta = pvdeg.weather.get(database="PVGIS", id=(45, -115))

In [13]:
fusion.store_single(
    weather_df=weather, meta=meta, tmy=True, source_name="pvgis"
)

coercing tmy data to year 1979


In [16]:
client = pvdeg.geospatial.start_dask()
coords = [(45 + i, -115 + k) for i in range(5) for k in range(5)]
geo_weather, geo_meta, failed = pvdeg.weather.weather_distributed(
    database="PVGIS", coords=coords,
)
client.close()

Dashboard: http://127.0.0.1:8787/status
Connected to a Dask scheduler | Dashboard: http://127.0.0.1:8787/status


In [17]:
for i in range(25):
    w = geo_weather.isel(gid=i).drop_vars(("gid",)).to_pandas()
    m = geo_meta.iloc[i].to_dict()

    fusion.store_single(
        weather_df=w, meta=m, tmy=True, source_name="pvgis"
    )

coercing tmy data to year 1979
duplicate file detected, skipping insert
metadata of duplicate file {'latitude': 45.0, 'longitude': -115.0, 'irradiance_time_offset': 0.0, 'altitude': 1948.0, 'wind_height': 10, 'Source': 'PVGIS'}
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979


We can easily get one output from what we have written so far. We need to be able to read MANY into dataset form.

In [18]:
fusion.sources()

{'pvgis': 25, 'solrad': 1}

In [19]:
lw, lm = fusion.load_single(
    latitude=sm["latitude"] + 10,
    longitude=sm["longitude"],
    source_name="solrad",
)

lw.shape

(7200, 22)

In [20]:
gw, gm = fusion.load_many(
    source_name="pvgis",
    spatial_search=True,
    spatial_search_distance_floor=140 * 1000,
    spatial_search_latitude0=45,
    spatial_search_longitude0=-114,
)

In [21]:
gm

,latitude,longitude,irradiance_time_offset,altitude,wind_height,Source
36,45.0,-114.0,0.0,2076.0,10,PVGIS
38,45.0,-112.0,0.0,2219.0,10,PVGIS
45,47.0,-115.0,0.0,1515.0,10,PVGIS
47,47.0,-113.0,0.0,1281.0,10,PVGIS
56,49.0,-114.0,0.0,1549.0,10,PVGIS
59,49.0,-111.0,0.0,1056.0,10,PVGIS


In [22]:
gm

,latitude,longitude,irradiance_time_offset,altitude,wind_height,Source
36,45.0,-114.0,0.0,2076.0,10,PVGIS
38,45.0,-112.0,0.0,2219.0,10,PVGIS
45,47.0,-115.0,0.0,1515.0,10,PVGIS
47,47.0,-113.0,0.0,1281.0,10,PVGIS
56,49.0,-114.0,0.0,1549.0,10,PVGIS
59,49.0,-111.0,0.0,1056.0,10,PVGIS
